## What is a pipeline

## Column Transformer

A Column Transformer is a powerful class in Scikit-Learn (sklearn.compose.ColumnTransformer) that allows you to apply different preprocessing steps to different columns of your dataset simultaneously.

In a real-world dataset, you rarely treat every column the same way. You might have numerical columns that need scaling and categorical columns that need encoding. A Column Transformer "bundles" these different operations into a single object.`

When is it used?

It is used during the Data Preprocessing phase of a machine learning pipeline, specifically when:

- Handling Mixed Data Types: You have a mix of numerical, categorical, and text data.

- Building Pipelines: You want to ensure your preprocessing steps are reproducible and can be exported as a single file alongside your model.

- Selective Scaling: You want to scale only certain numerical features while leaving binary (0/1) features untouched.

How it works (The Logic)

Instead of manually splitting your dataframe, transforming parts, and gluing them back together, you define a list of tuples:

Name: A nickname for the step (e.g., "cat_encode").

Transformer: The actual object (e.g., OneHotEncoder()).

Columns: The list of columns to apply this to.

Advantages (Pros)

Code Cleanliness: It eliminates the need for manual "slice-and-dice" operations on your DataFrames.

Prevents Data Leakage: Because it integrates into a Scikit-Learn Pipeline, it ensures that your transformers are fitted only on the training data and then applied to the test data.

Parallel Processing: It processes the column groups independently, which is more efficient than sequential code.

"Remainder" Control: It has a remainder parameter that allows you to easily decide whether to drop columns you didn't transform or pass them through as-is.

Single Output: It automatically concatenates all transformed columns back into a single NumPy array or DataFrame.

Disadvantages (Cons)

Output Format: By default, it returns a NumPy array, which strips away the original pandas column names (making it harder to inspect the data immediately after).

Complexity for Beginners: The syntax of nested tuples and lists can be intimidating if you are just starting with Scikit-Learn.

Sparse Matrix Issues: If one of your transformers (like One-Hot Encoding) produces a sparse matrix, the entire output may become a sparse matrix, which can be tricky to work with if you aren't expecting it.

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline # Corrected import for Pipeline

# 1. Create a sample DataFrame
data = {
    'Age': [25, 30, 35, 40, 45, None, 55, 60, 65, 70],
    'Salary': [50000, 60000, 75000, None, 90000, 100000, 110000, 120000, 130000, 140000],
    'City': ['New York', 'London', 'Paris', 'New York', 'London', 'Paris', 'London', 'New York', 'Paris', 'London'],
    'Gender': ['Male', 'Female', 'Male', 'Female', 'Male', 'Female', 'Male', 'Female', 'Male', 'Female'],
    'Experience': [2, 5, 8, 10, 12, 15, 18, 20, 22, 25]
}
df = pd.DataFrame(data)
display(df.head())

# 2. Define the types of columns
numerical_cols = ['Age', 'Salary', 'Experience']
categorical_cols = ['City', 'Gender']

# 3. Create preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Handle missing values by imputation
    ('scaler', StandardScaler())  # Scale numerical features
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encode categorical features
])

# 4. Create a ColumnTransformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'  # Keep other columns (if any) that are not transformed
)

# 5. Apply the ColumnTransformer to the DataFrame
df_transformed = preprocessor.fit_transform(df)

# Display the shape of the transformed data
print(f"\nShape of original DataFrame: {df.shape}")
print(f"Shape of transformed data: {df_transformed.shape}")

# To see the transformed data, you can convert it back to a DataFrame (optional, for inspection)
# Get feature names after one-hot encoding
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names) # No 'remainder_col_if_any' needed if remainder is passthrough for existing cols

# Create a DataFrame from the transformed array with correct column names
df_transformed_named = pd.DataFrame(df_transformed, columns=all_feature_names)
print("\nTransformed data (first 5 rows) with column names:\n", df_transformed_named.head())

,Age,Salary,City,Gender,Experience
0,25.0,50000.0,New York,Male,2
1,30.0,60000.0,London,Female,5
2,35.0,75000.0,Paris,Male,8
3,40.0,NaN,New York,Female,10
4,45.0,90000.0,London,Male,12



Shape of original DataFrame: (10, 5)
Shape of transformed data: (10, 8)

Transformed data (first 5 rows) with column names:
         Age    Salary  Experience  City_London  City_New York  City_Paris  \
0 -1.549969 -1.695665   -1.625470          0.0            1.0         0.0   
1 -1.201226 -1.336583   -1.208683          1.0            0.0         0.0   
2 -0.852483 -0.797960   -0.791896          0.0            0.0         1.0   
3 -0.503740  0.000000   -0.514038          0.0            1.0         0.0   
4 -0.154997 -0.259337   -0.236179          1.0            0.0         0.0   

   Gender_Female  Gender_Male  
0            0.0          1.0  
1            1.0          0.0  
2            0.0          1.0  
3            1.0          0.0  
4            0.0          1.0  


## Function Transformer

A Function Transformer (sklearn.preprocessing.FunctionTransformer) is a tool in Scikit-Learn that allows you to turn any Python function into a formal "Transformer" object.

Essentially, it acts as a wrapper. It takes a mathematical operation you've written (like a log transform or a custom calculation) and makes it compatible with Scikit-Learn’s fit() and transform() methods.

It is used during Feature Engineering when you need to perform "stateless" transformations. "Stateless" means the operation doesn't need to learn anything from the data (unlike a Scaler that needs to learn the mean and standard deviation).

Common use cases include:

- Mathematical Scaling: Applying $log(x)$, $exp(x)$, or $x^2$ to change the distribution of a feature.

- Feature Creation: Calculating the length of a string in a "Comments" column.

- Cleaning: Stripping whitespace from strings or replacing specific values based on a custom rule.

- Custom Binning: Grouping ages into "Young," "Adult," and "Senior" using a custom Python function.

Advantages (Pros)

Pipeline Integration: It allows you to include custom logic inside a Pipeline. This ensures that your custom cleaning steps are applied automatically to the test data without manual intervention.

Simplicity: If you already have a Python function that cleans your data, you don't need to write a complex custom class; you just wrap it in a FunctionTransformer.

Reproducibility: By keeping the logic inside the transformer, you ensure that the exact same math is applied to every new piece of data your model sees.

Flexibility: It can handle any transformation you can express in code, making it the "Swiss Army Knife" of preprocessing.

Disadvantages (Cons)

Stateless Only: It cannot "remember" data. For example, you cannot use it for Standard Scaling because it doesn't have a way to store the mean of the training set to use on the test set. (For that, you would need to build a custom class with a fit method).

Serialization Issues: If you use a lambda function inside the transformer, you might have trouble saving (pickling) your model for deployment later.

Debugging Complexity: If the function fails, the error message might come from deep inside your custom code rather than the Scikit-Learn library, making it slightly harder to debug within a large pipeline.

Different types of Function Transformer are:

- Log Transform

- Reciprocal Transform

- Square Root Transform

### Log Transformation

The Log transformation applies the natural logarithm to the data. In practice, we use $log(1+x)$ (known as log1p) to avoid errors with zero values.

- What it is: Compresses the range of values, specifically "pulling in" long right tails.

- When to use: When your data is right-skewed (Positive Skew) and has a wide range of values (e.g., Salary, House Prices, Population).

- Pros: Very effective at handling exponential growth and outliers.

- Cons: Cannot handle negative numbers; sensitive to zero (unless using $1+x$).

In [7]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

# Log transform (log1p handles 0s)
log_tf = FunctionTransformer(np.log1p)
data_log = log_tf.transform([[10], [100], [1000]])

data_log

array([[2.39789527],
       [4.61512052],
       [6.90875478]])

### Reciprocal Transformation

This transformation involves dividing 1 by the original value ($1/x$).

- What it is: It drastically shrinks large values and expands small values near zero. It essentially "flips" the relationship.

- When to use: When you want to penalize large values heavily or when the ratio is more meaningful (e.g., transforming "Time to complete a task" into "Speed").

- Pros: Good for extreme right-skewed data.

- Cons: It cannot handle zero (division by zero error). It reverses the order of the data (large becomes small), which can make interpretation confusing.

In [9]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

# Reciprocal transform using np.reciprocal for element-wise operation
reciprocal_tf = FunctionTransformer(np.reciprocal)
data_rec = reciprocal_tf.transform([[2], [4], [10]])

data_rec

array([[0],
       [0],
       [0]])

### Square Root Transformation

This transformation applies the square root ($\sqrt{x}$) to each data point.

- What it is: A moderate transformation that reduces skewness but is less aggressive than the Log transform.

- When to use: When your data is right-skewed but the range is not extreme (e.g., count data like "Number of items sold").

- Pros: Can handle zero values (unlike Log or Reciprocal). It maintains the original scale's "feeling" better than Log.

- Cons: Not strong enough for very heavily skewed data with massive outliers.

In [10]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

# Square Root transform
sqrt_tf = FunctionTransformer(np.sqrt)
data_sqrt = sqrt_tf.transform([[4], [16], [25]])

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

# 1. Create a dummy dataset
data = {
    'Age': [25, 30, 35, 40],
    'Salary': [50000, 80000, 120000, 500000], # Heavily skewed
    'City': ['New York', 'London', 'London', 'Paris']
}
df = pd.DataFrame(data)

# 2. Define a custom function
# Let's say we want to categorize salary as "High" or "Low" based on a threshold
def salary_thrasher(x):
    return np.where(x > 100000, 1, 0)

# 3. Create the FunctionTransformers
log_transformer = FunctionTransformer(np.log1p)
custom_bin_transformer = FunctionTransformer(salary_thrasher)

# 4. Integrate into a ColumnTransformer
# This applies the log to 'Salary' and our custom binning logic to 'Salary' again
preprocessor = ColumnTransformer(transformers=[
    ('log_transform', log_transformer, ['Salary']),
    ('is_high_salary', custom_bin_transformer, ['Salary'])
], remainder='passthrough')

# 5. Transform the data
transformed_data = preprocessor.fit_transform(df)

# Convert back to DataFrame for readability
columns = ['Log_Salary', 'Is_High_Salary', 'Age', 'City']
transformed_df = pd.DataFrame(transformed_data, columns=columns)

print("Original Data:")
print(df)
print("\nTransformed Data:")
print(transformed_df)

Original Data:
   Age  Salary      City
0   25   50000  New York
1   30   80000    London
2   35  120000    London
3   40  500000     Paris

Transformed Data:
  Log_Salary Is_High_Salary Age      City
0  10.819798              0  25  New York
1  11.289794              0  30    London
2  11.695255              1  35    London
3  13.122365              1  40     Paris


## Power Transformer

A Power Transformer (sklearn.preprocessing.PowerTransformer) is a preprocessing technique used to map data from any distribution to a Gaussian (Normal) distribution (a bell curve).

Unlike a Function Transformer, which applies a fixed mathematical formula (like $log$), a Power Transformer is parametric. This means it analyzes your data to find the optimal power parameter (called $\lambda$ or "lambda") that minimizes skewness and stabilizes the variance.

It is used during the Feature Engineering phase, particularly when:

- The model assumes normality: Many algorithms, such as Linear Regression, Logistic Regression, LDA, and Gaussian Naive Bayes, perform significantly better when features are normally distributed.

- The data is heavily skewed: When you have "long tails" in your data that a simple log transform can't fully fix.

- Variance is inconsistent: When the "spread" of your data changes across its range (Heteroscedasticity).

- Handling both positive and negative values: It can handle datasets that contain zeros or negative numbers (using the Yeo-Johnson method).

How it works: The Two Methods

The Power Transformer usually offers two internal algorithms:

Box-Cox Transform:

- Condition: Requires the data to be strictly positive (all values > 0).

- Formula: $y = \frac{x^\lambda - 1}{\lambda}$

Yeo-Johnson Transform:

- Condition: Works with both positive, zero, and negative values.

- Use Case: This is the default and most flexible option in Scikit-Learn.

Fromula:

For a given input $x$ and the transformation parameter $\lambda$, the transformed value $y^{(\lambda)}$ is calculated as:$$y^{(\lambda)} =
\begin{cases}
\frac{(x + 1)^\lambda - 1}{\lambda} & \text{if } x \geq 0, \lambda \neq 0 \\
\ln(x + 1) & \text{if } x \geq 0, \lambda = 0 \\
-\frac{(-x + 1)^{2 - \lambda} - 1}{2 - \lambda} & \text{if } x < 0, \lambda \neq 2 \\
-\ln(-x + 1) & \text{if } x < 0, \lambda = 2
\end{cases}$$

Pros (Advantages)
Automatic Optimization: You don't have to guess whether to use log, square root, or reciprocal; the algorithm finds the best power for you.

Handles Multi-directional Skew: It can fix both right-skewed and left-skewed data.

Improves Model Performance: For distance-based and linear models, it often leads to higher accuracy and more stable coefficients.

Standardization included: By default, Scikit-Learn's PowerTransformer also applies Zero-mean and Unit-variance scaling (Standardization) after the transform.

Cons (Disadvantages)

- Loss of Interpretability: Once you transform a feature like "Price" using PowerTransformer, the resulting value (e.g., 1.42) no longer has a direct real-world meaning.

- Computational Cost: Because it has to "search" for the best $\lambda$ through optimization, it is slower than a simple FunctionTransformer.

- Not for Trees: It is generally unnecessary for tree-based models (Random Forest, XGBoost), which are invariant to the distribution of features.

- Sensitive to Outliers: While it reduces the impact of skewness, extreme outliers can still heavily influence the calculation of the optimal $\lambda$.

In [11]:
from sklearn.preprocessing import PowerTransformer
import numpy as np

# Sample skewed data
X = [[1], [10], [100], [1000]]

# Initialize (Yeo-Johnson is default)
pt = PowerTransformer(method='yeo-johnson')

# Fit and transform
X_transformed = pt.fit_transform(X)

print(X_transformed)
# The output will be centered around 0 and look normally distributed

[[-1.30817404]
 [-0.50141245]
 [ 0.45759942]
 [ 1.35198707]]


## Box Cox

The Box-Cox transform is the classic power transformation. It is highly effective but has one strict limitation: it only works on positive data.

- When to use: Use it when your numerical features are strictly positive ($x > 0$) and you want to achieve a perfectly symmetrical bell curve.

- The Formula: $y = \frac{x^\lambda - 1}{\lambda}$ (if $\lambda \neq 0$).

- Pros: Very mathematically stable; often achieves a better "normal" shape than simple log transforms.

- Cons: Strictly requires positive values. If your data contains $0$ or negative numbers, it will throw an error.

In [12]:
from sklearn.preprocessing import PowerTransformer
import numpy as np

# Sample positive data (strictly > 0)
X = np.array([[10], [50], [200], [1000]])

# Initialize Box-Cox
pt_boxcox = PowerTransformer(method='box-cox')

# Fit and Transform
X_transformed = pt_boxcox.fit_transform(X)
print(f"Optimal Lambda: {pt_boxcox.lambdas_}")

Optimal Lambda: [1.16752605e-09]


## Yeo-Johnson Transformation

The Yeo-Johnson transform is a modern extension of Box-Cox. It was created to overcome the "positive only" limitation.

- When to use: This is the default choice for most data scientists. Use it when your data contains zeros or negative numbers, or if you just want a "one-size-fits-all" solution.

- The Formula: A complex piece-wise function that handles $x \geq 0$ and $x < 0$ separately.

- Pros: Extremely flexible; handles negative and zero values; handles both right and left skew.

- Cons: Slightly more complex to calculate; the results can be slightly less "clean" than Box-Cox if the data is purely positive

In [13]:
from sklearn.preprocessing import PowerTransformer
import numpy as np

# Sample data with zeros and negative values
X = np.array([[-10], [0], [50], [500]])

# Initialize Yeo-Johnson (Default)
pt_yeo = PowerTransformer(method='yeo-johnson')

# Fit and Transform
X_transformed = pt_yeo.fit_transform(X)
print(f"Optimal Lambda: {pt_yeo.lambdas_}")

Optimal Lambda: [0.46688041]
